<a href="https://colab.research.google.com/github/jacquiline18/Jacquiline-CodeBooster-Internship-2026-Phase_01-Data_Engineering-/blob/main/Day_03_ETL_and_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
row_df = pd.read_csv('messy_sales_data.csv')

print('*' * 55)
print('DATA QUALITY DIAGNOSIS REPORT')
print('*' * 55)
print('\n[1] MISSING VALUES per column')
print(row_df.isnull().sum())
print(f'\n[2] DUPLICATE ROWS: {row_df.duplicated().sum()}')
print('\n[2] DATA TYPES')
print(row_df.dtypes)
print(f'\n[3] UNIQUE CATEGORIES: {row_df["category"].unique()}')
print(f'[4] Sample customer names: {row_df["customer_name"].dropna().unique()}')
print(f'[4] Sample order_data value: {row_df["order_date"].dropna().unique()}')

*******************************************************
DATA QUALITY DIAGNOSIS REPORT
*******************************************************

[1] MISSING VALUES per column
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS: 0

[2] DATA TYPES
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[3] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]
[4] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer' 'Pooja Gupta' 'SURESH RAO'
 'Meera Joshi' 'Arjun Nair' 'Tanvi Mehta' 'Kiran Mehta' 'Rohit Verma'
 'Sneha Reddy' 'Gaurav Shukla' 'Nisha Kapoor' 'Ajay Tiwari' 'ANANYA D

In [6]:
df = row_df.copy()
print(f'working copy created:{df.shape}')
print('row_df is untouched - we can always reset by running df = row_df.copy()')

working copy created:(30, 9)
row_df is untouched - we can always reset by running df = row_df.copy()


In [10]:
df['customer_name'] = df['customer_name'].fillna('Unknown Customer')
median_qty = df['quantity'].median()
df['quantity'] = df['quantity'].fillna(median_qty)
print(f'Filled missing quantity with median: {median_qty}')

Filled missing quantity with median: 2.0


In [13]:
print(f'Before duplication: {len(df)} rows')
print(f'Duplicate rows: {df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name']])
df.drop_duplicates(inplace=True)
print(f'After duplication: {len(df)} rows')
print(f'Rows remove:{len(row_df)-len(df)}')


Before duplication: 30 rows
Duplicate rows: 0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name]
Index: []
After duplication: 30 rows
Rows remove:0


In [15]:
print('Sample dates before parsing:')
print(df['order_date'].head(8))

df['order_date'] = pd.to_datetime(
    df['order_date'],
    dayfirst=True,
    errors='coerce'
)

nat_count = df['order_date'].isnull().sum()
print(f'\nUnparseable dates (NaT): {nat_count}')

df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['day'] = df['order_date'].dt.day

Sample dates before parsing:
0    2024-01-05
1    2024-01-07
2    2024-01-08
3    2024-01-10
4    2024-01-05
5    07-01-2024
6    2024-01-12
7    2024-01-13
Name: order_date, dtype: object

Unparseable dates (NaT): 17


In [17]:
print('before standardization:',df['customer_name'].unique()[:6])
df['customer_name'] = df['customer_name'].str.strip()

before standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']


In [19]:
wrong_mask = (df['product'] == 'Keyboard') & (df['category'].isnull())
print(df[wrong_mask][['product', 'category']])

Empty DataFrame
Columns: [product, category]
Index: []


In [21]:
df['quantity']=pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price']=pd.to_numeric(df['unit_price'], errors='coerce')
df['revenue']=df['quantity']*df['unit_price']
print('Revenue column created')
print(df[['customer_name','product','quantity','unit_price','revenue']].head(5))

Revenue column created
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop       2.0       45000  90000.0
1    Priya Nair       NaN       1.0       15000  15000.0
2    AMIT VERMA  Keyboard       3.0        1200   3600.0
3  Sunita Patel   Monitor       2.0       22000  44000.0
4  Ramesh Kumar    Laptop       2.0       45000  90000.0


In [4]:
import pandas as pd
row_df = pd.read_csv('/content/messy_sales_data.csv')

# Initialize df from row_df to avoid NameError if previous cells haven't run
# Note: For this to be a true 'POST-CLEANING' report, previous cleaning cells must be executed.
df = row_df.copy()

print('='*55)
print('POST-CLEANING VALIDATION REPORT')
print('='*55)

print(f'Original rows: {len(row_df)}')
print(f'Cleaned rows: {len(df)}')

print('\nMissing values after cleaning:')
print(df.isnull().sum())

POST-CLEANING VALIDATION REPORT
Original rows: 30
Cleaned rows: 30

Missing values after cleaning:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64


In [7]:
import pandas as pd

# Ensure revenue column exists by re-creating it if necessary
df['quantity']=pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price']=pd.to_numeric(df['unit_price'], errors='coerce')
df['revenue']=df['quantity']*df['unit_price']

product_rev = (
    df.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values(by='revenue', ascending=False)
)
print('Revenue by product:')
display(product_rev)

Revenue by product:


,product,revenue
2,Laptop,450000.0
3,Monitor,110000.0
0,Headphones,28000.0
4,Mouse,20800.0
1,Keyboard,20400.0
5,USB Hub,19800.0
6,Webcam,15000.0


In [8]:
df.to_csv('clean_sales_data_dat.csv', index=False)
print('Cleaned data saved to clean_sales_data_dat.csv')

Cleaned data saved to clean_sales_data_dat.csv


What are the 3 stagers of ETL ? Describe each stage using an eapmle from today sales dataset

## The 3 Stages of ETL (Extract, Transform, Load)

ETL is a fundamental process in data warehousing and data integration. It involves three key stages:

### 1. Extract

**Description:** This stage involves retrieving data from various source systems. Data can come from databases, files (like CSV, Excel, XML), APIs, or other applications. The primary goal is to collect all necessary raw data.

**Example from `sales_data`:**
In our case, the **Extract** stage involved loading the `messy_sales_data.csv` file into a pandas DataFrame. This is where we pulled the raw, uncleaned sales records from the CSV file.

```python
row_df = pd.read_csv('/content/messy_sales_data.csv')
```

### 2. Transform

**Description:** This is the most crucial stage, where the extracted data is cleaned, validated, filtered, enriched, and aggregated to fit the requirements of the target system (e.g., a data warehouse or for analysis). This involves a series of operations like:
*   Handling missing values
*   Correcting data types
*   Removing duplicates
*   Standardizing formats
*   Deriving new features
*   Aggregating data

**Example from `sales_data`:**
We performed several **Transform** operations on our `sales_data`:
*   **Handling Missing Values:** We filled missing `customer_name` values with 'Unknown Customer' and `quantity` with its median.
    ```python
df['customer_name'] = df['customer_name'].fillna('Unknown Customer')
median_qty = df['quantity'].median()
df['quantity'] = df['quantity'].fillna(median_qty)
    ```
*   **Data Type Conversion & Standardization:** We converted `order_date` to datetime objects, handling unparseable dates, and then extracted `year`, `month`, and `day`. We also converted `quantity` and `unit_price` to numeric types.
    ```python
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True, errors='coerce')
df['quantity']=pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price']=pd.to_numeric(df['unit_price'], errors='coerce')
    ```
*   **Cleaning Text Data:** We removed leading/trailing whitespace from `customer_name`.
    ```python
df['customer_name'] = df['customer_name'].str.strip()
    ```
*   **Feature Engineering:** We created a new `revenue` column by multiplying `quantity` and `unit_price`.
    ```python
df['revenue']=df['quantity']*df['unit_price']
    ```
*   **Aggregation:** We aggregated sales data by product to get `product_rev`.
    ```python
product_rev = df.groupby('product')['revenue'].sum().reset_index().sort_values(by='revenue', ascending=False)
    ```

### 3. Load

**Description:** In this final stage, the transformed data is moved from the staging area into the final target system. This can be a data warehouse, a data mart, or another operational data store. The loading process can be a full load (replacing all existing data) or an incremental load (adding or updating only new/changed data).

**Example from `sales_data`:**
After all the cleaning and transformation, we **Loaded** our cleaned `df` into a new CSV file named `clean_sales_data_dat.csv`. This new file represents the data ready for further analysis or to be loaded into a data warehouse.

```python
df.to_csv('clean_sales_data_dat.csv', index=False)
```

a dataframe has 500 rows, After calling df.dropna().it has 412 rows.what does this tell you?

This tells us that out of the original 500 rows, 88 rows (500 - 412 = 88) contained at least one missing (NaN) value in one or more of their columns. The `df.dropna()` function removed all these rows because it defaults to dropping rows with *any* missing values.

In [ ]:
write code to remove duplicates from df where same row means customer name